# seisfetch: Quickstart

This notebook walks through the core API of `seisfetch`:

1. Fetch waveforms from archive-backed clients first, with FDSN as a fallback
2. Decode directly to numpy without ObsPy in the download path
3. Convert to xarray and hand sparse sensor summaries into Earth2Studio-style pipelines
4. Keep raw miniSEED available for custom downstream processing


## Install

```bash
pip install -e "."                              # core only
pip install -e ".[xarray,obspy]"                # xarray + ObsPy interop
pip install -e ".[auth]" earthscope-cli         # EarthScope S3 auth support
```


In [ ]:
from datetime import datetime

import numpy as np

from seisfetch import (
    FDSNClient,
    SeisfetchClient,
    SeismicDataFrameSource,
    bundle_to_xarray,
    list_providers,
    parse_mseed,
    route_network,
)

## 1. Fetch from S3 → numpy

The `s3_open` backend provides **anonymous (no credentials) access to SCEDC and NCEDC**.

> **EarthScope** (`earthscope-geophysical-data`) currently requires `backend="s3_auth"`
> with `earthscope-sdk` credentials. Per the EarthScope SDK docs, direct S3 access is
> still gated and your account may need explicit approval from `data-help@earthscope.org`.

Times can be ISO strings, epoch floats, or `datetime` objects (no ObsPy needed).


In [ ]:
client = SeisfetchClient(backend="s3_open")

# CI.ABL → auto-routes to SCEDC (open, anonymous access)
# For EarthScope networks (IU, TA, UW, ...) use backend="s3_auth" (requires earthscope-sdk)
bundle = client.get_numpy(
    "CI",
    "ABL",
    channel="BHZ",
    starttime="2024-01-15T00:00:00",
    endtime="2024-01-15T00:10:00",
)

print(f"Traces: {len(bundle)}")
print(f"Channel IDs: {bundle.ids}")

# Get as a plain dict of numpy arrays
arrays = bundle.to_dict()
for nslc, data in arrays.items():
    print(f"  {nslc}: shape={data.shape}, dtype={data.dtype}")

## 1b. EarthScope SDK credentials and authenticated S3

For EarthScope networks like `IU`, `TA`, and `UW`, use `backend="s3_auth"`.

1. Install auth support: `pip install -e ".[auth]" earthscope-cli`
2. Log in once on this machine: `es login`
3. If S3 access is still denied, ask EarthScope to enable direct S3 access for your account: `data-help@earthscope.org`
4. Verify the SDK can vend temporary AWS credentials before using the client


In [ ]:
# Requires: `es login` and an EarthScope account enabled for direct S3 access
try:
    auth_client = SeisfetchClient(backend="s3_auth")
    bundle_auth = auth_client.get_numpy(
        "IU",
        "ANMO",
        location="00",
        channel="BHZ",
        starttime="2024-01-15T00:00:00",
        endtime="2024-01-15T00:10:00",
    )
    print(f"IU.ANMO via EarthScope S3 auth: {len(bundle_auth)} traces")
    print(bundle_auth.ids)
except ImportError as exc:
    print('Install auth support with: pip install -e ".[auth]" earthscope-cli')
    print(exc)
except Exception as exc:
    print("EarthScope auth is not ready on this machine.")
    print("Run `es login` and confirm your account is enabled for direct S3 access.")
    print(type(exc).__name__, exc)

## 2. Auto-routing by network code

The client automatically picks the fastest S3 bucket for each network:

| Network | Datacenter | S3 Bucket | Auth |
|---------|-----------|-----------|------|
| CI | SCEDC | scedc-pds (us-west-2) | None (open) |
| BK | NCEDC | ncedc-pds (us-east-2) | None (open) |
| IU, UW, TA, ... | EarthScope | earthscope-geophysical-data (us-east-2) | `earthscope-sdk` required |

In [ ]:
# Check routing for any network
for net in ["CI", "BK", "NC", "IU", "UW", "TA"]:
    print(f"  {net} → {route_network(net)}")

In [ ]:
# SCEDC and NCEDC are per-channel archives — specify channel explicitly
bundle_ci = client.get_numpy(
    "CI",
    "SDD",
    channel="BHZ",
    starttime="2024-06-01T00:00:00",
    endtime="2024-06-01T00:10:00",
)
print(f"CI.SDD.BHZ: {bundle_ci.to_dict().get('CI.SDD..BHZ', np.array([])).shape}")

## 3. FDSN web services as a fallback

The preferred order in this notebook is:

1. `s3_open` for SCEDC and NCEDC
2. `s3_auth` for EarthScope
3. `fdsn` only when the archive-backed path is unavailable or the network is not
   served from those buckets

The helper below enforces that archive-first policy.


In [ ]:
def get_numpy_archive_first(
    network,
    station,
    *,
    starttime,
    endtime,
    location="*",
    channel="*",
    fallback_provider=None,
):
    dc = route_network(network)
    primary_backend = "s3_open" if dc in {"scedc", "ncedc"} else "s3_auth"

    try:
        kwargs = {"datacenter": dc} if primary_backend == "s3_open" else {}
        primary = SeisfetchClient(backend=primary_backend, **kwargs)
        bundle = primary.get_numpy(
            network,
            station,
            location=location,
            channel=channel,
            starttime=starttime,
            endtime=endtime,
        )
        if len(bundle):
            print(f"archive path: {primary_backend} ({dc})")
            return bundle
    except Exception as exc:
        print(f"archive path failed: {type(exc).__name__}: {exc}")

    if fallback_provider is None:
        raise RuntimeError(
            "archive path failed and no FDSN fallback provider was supplied"
        )

    print(f"falling back to FDSN provider {fallback_provider}")
    fallback = SeisfetchClient(backend="fdsn", providers=fallback_provider)
    return fallback.get_numpy(
        network,
        station,
        location=location,
        channel=channel,
        starttime=starttime,
        endtime=endtime,
    )

In [ ]:
# Discover a GE.BHZ waveform window that can be used for FDSN fallback
ge_start = "2011-03-11T06:00:00"
ge_end = "2011-03-11T06:05:00"

providers = list_providers()
print("Fallback provider:", providers["GEOFON"])

geofon_meta = FDSNClient("GEOFON")
station_text = geofon_meta.get_station_text(
    network="GE",
    channel="BHZ",
    starttime=ge_start,
    endtime=ge_end,
    level="channel",
    format="text",
)

lines = [line for line in station_text.splitlines() if line]
header = lines[0].lstrip("#").split("|")
rows = [dict(zip(header, line.split("|"))) for line in lines[1:]]
if not rows:
    raise RuntimeError("No GE.BHZ channels found for the GEOFON fallback window")

ge_row = None
ge_loc = None
for row in rows:
    loc = row.get("Location", "") or "*"
    try:
        raw_probe = geofon_meta.get_raw(
            row["Network"],
            row["Station"],
            location=loc,
            channel=row["Channel"],
            starttime=ge_start,
            endtime=ge_end,
        )
    except Exception:
        continue
    if raw_probe:
        ge_row = row
        ge_loc = loc
        break

if ge_row is None:
    raise RuntimeError(
        "No GE.BHZ waveform data available from GEOFON for the fallback window"
    )

print(
    f"Validated fallback example {ge_row['Network']}.{ge_row['Station']}.{ge_loc}.{ge_row['Channel']} "
    f"for {ge_start} to {ge_end}"
)

In [ ]:
# Archive-first fetch with FDSN fallback for a non-archive example
bundle_ge = get_numpy_archive_first(
    ge_row["Network"],
    ge_row["Station"],
    location=ge_loc,
    channel=ge_row["Channel"],
    starttime=ge_start,
    endtime=ge_end,
    fallback_provider="GEOFON",
)
print(f"Fallback fetch: {len(bundle_ge)} traces")
print(bundle_ge.ids)

## 4. Output: xarray Dataset

Convert any TraceBundle to an xarray Dataset with `datetime64[ns]` coordinates.
Requires: `pip install xarray`


In [ ]:
ds = bundle_to_xarray(bundle)
print(ds)

# Each channel is a DataArray with time coordinate
for var in ds.data_vars:
    da = ds[var]
    print(f"\n{var}:")
    print(f"  shape: {da.shape}")
    print(f"  time range: {da.time.values[0]} → {da.time.values[-1]}")
    print(f"  sampling_rate: {da.attrs['sampling_rate']} Hz")

## 5. Output: xarray via client shortcut


In [ ]:
ds_direct = client.get_xarray(
    "CI",
    "ABL",
    channel="BHZ",
    starttime="2024-01-15T00:00:00",
    endtime="2024-01-15T00:10:00",
)
print(ds_direct)

## 6. Earth2Studio sparse sensor handoff

`SeismicDataFrameSource` turns a waveform window into one row per station-channel
observation with summary statistics. This is the natural bridge into sparse sensor
and data-assimilation workflows in Earth2Studio-style pipelines.


In [ ]:
# Build a sparse-sensor table from the waveform window
df_source = SeismicDataFrameSource(bundle)
df = df_source(datetime(2024, 1, 15), list(ds.data_vars))

columns = [
    "time",
    "variable",
    "network",
    "station",
    "location",
    "channel",
    "sampling_rate",
    "amplitude_rms",
    "amplitude_max",
    "num_samples",
]
df[columns]

## 7. Output: ObsPy Stream (optional interop)

When you need ObsPy processing (filtering, response removal, plotting),
convert the `TraceBundle` after download.

Note: downloading and decoding still uses S3 + pymseed — ObsPy only
comes in after the data is local.


In [ ]:
try:
    from seisfetch import bundle_to_obspy

    st = bundle_to_obspy(bundle)
    print(st)
    # st.filter("bandpass", freqmin=0.1, freqmax=2.0)
    # st.plot()
except ImportError:
    print("ObsPy not installed — skipping.  pip install obspy")

## 8. Parse local miniSEED files

`parse_mseed()` works on any raw bytes — from S3, HTTP, or local disk.


In [ ]:
# Simulate reading a local file
raw_bytes = client.get_raw(
    "CI",
    "ABL",
    channel="BHZ",
    starttime="2024-01-15",
    endtime="2024-01-15T00:01:00",
)
print(f"Raw miniSEED: {len(raw_bytes):,} bytes")

# Parse → numpy
bundle_local = parse_mseed(raw_bytes)
print(f"Parsed: {len(bundle_local)} traces, IDs: {bundle_local.ids}")

# To parse a real file:
# with open("data.mseed", "rb") as f:
#     bundle = parse_mseed(f.read())

## 9. Raw bytes for custom pipelines

If you want to handle miniSEED yourself (e.g., feed to a different decoder),
`get_raw()` gives you the exact bytes from the provider.


In [ ]:
raw = client.get_raw(
    "CI",
    "ABL",
    channel="BHZ",
    starttime="2024-01-15T00:00:00",
    endtime="2024-01-15T00:01:00",
)
print(f"Raw bytes: {len(raw):,}")
print(f"First 20 bytes: {raw[:20]}")

## Summary

| Method | Returns | Deps |
|--------|---------|------|
| `get_raw()` | `bytes` | core |
| `get_numpy()` | `TraceBundle` → `.to_dict()` = `{nslc: ndarray}` | core |
| `get_xarray()` | `xarray.Dataset` | + xarray |
| `SeismicDataFrameSource(...)` | sparse sensor `pandas.DataFrame` | + pandas, xarray |
| `get_waveforms()` | `obspy.Stream` | + obspy |
| `parse_mseed(bytes)` | `TraceBundle` | core |
